# 02 — Connexion aux bases de données

Ce notebook montre **toutes les techniques de connexion** couvertes par le repo :
- Supabase (SDK officiel **et** connexion SQL directe)
- PostgreSQL classique
- MySQL
- SQLite

Chaque section montre le code, explique le "pourquoi", et gère les erreurs.


In [ ]:
import sys
sys.path.append("..")
from utils.logger import get_logger
logger = get_logger("notebook_02")


## 1. Supabase — via le SDK officiel (`supabase-py`)

Recommandé pour la majorité des usages : API REST auto-générée (PostgREST),
gestion de l'authentification, du Realtime et du Storage inclus.


In [ ]:
from db_connection.supabase_connector import get_supabase_client

supabase_client = get_supabase_client()
print("Client Supabase prêt :", supabase_client)


## 2. Supabase — connexion SQL directe (SQLAlchemy)

Utile quand tu as besoin de requêtes SQL complexes (jointures avancées,
fonctions PostgreSQL natives) que l'API REST ne permet pas facilement.


In [ ]:
from db_connection.supabase_connector import get_supabase_sql_engine

supabase_engine = get_supabase_sql_engine()
print("Moteur SQL Supabase prêt :", supabase_engine)


## 3. PostgreSQL classique

Deux façons : connexion brute `psycopg2` (curseur SQL direct) ou moteur
SQLAlchemy (pratique avec pandas / ORM).

Le connecteur inclut un **retry automatique** en cas d'échec temporaire
(base en redémarrage, réseau instable).


In [ ]:
from db_connection.postgresql_connector import get_pg_connection, get_pg_engine

# Connexion brute
pg_conn = get_pg_connection(max_retries=3, retry_delay=2.0)
print("Connexion PostgreSQL (psycopg2) OK.")
pg_conn.close()

# Moteur SQLAlchemy (avec pool de connexions)
pg_engine = get_pg_engine(pool_size=5, max_overflow=10)
print("Moteur PostgreSQL (SQLAlchemy) prêt :", pg_engine)


## 4. MySQL

Même logique que PostgreSQL : connexion brute ou moteur SQLAlchemy (via PyMySQL).


In [ ]:
from db_connection.mysql_connector import get_mysql_connection, get_mysql_engine

mysql_conn = get_mysql_connection(max_retries=3)
print("Connexion MySQL OK.")
mysql_conn.close()

mysql_engine = get_mysql_engine()
print("Moteur MySQL prêt :", mysql_engine)


## 5. SQLite

Idéal pour prototyper sans serveur : la base est un simple fichier local,
créé automatiquement s'il n'existe pas.


In [ ]:
from db_connection.sqlite_connector import get_sqlite_connection, get_sqlite_engine

sqlite_conn = get_sqlite_connection()
print("Connexion SQLite OK.")
sqlite_conn.close()

sqlite_engine = get_sqlite_engine()
print("Moteur SQLite prêt :", sqlite_engine)


## 6. Factory unique : `get_engine()`

Pour éviter d'importer manuellement le bon connecteur à chaque fois,
le repo fournit une factory unique dans `db_connection/connection_utils.py`.


In [ ]:
from db_connection.connection_utils import get_engine, DatabaseSession

# Une seule ligne pour obtenir n'importe quel moteur
engine = get_engine("sqlite")  # ou "postgresql", "mysql", "supabase"
print(engine)

# Ou en context manager, avec fermeture automatique et propre
with DatabaseSession("sqlite") as session_engine:
    print("Session ouverte :", session_engine)
# -> connexion fermée automatiquement ici


## Gestion des erreurs — ce qu'il faut retenir

Tous les connecteurs du repo :
- lèvent une `ConnectionError` explicite en cas d'échec (jamais une erreur brute peu claire),
- loggent chaque tentative (voir `utils/logger.py`),
- utilisent `pool_pre_ping=True` pour éviter les connexions "mortes" silencieuses,
- pour Postgres/MySQL : retry automatique configurable.

## Suite

➡️ Passe au notebook **`03_requetage.ipynb`** pour apprendre à interroger ces bases
(SELECT, INSERT, UPDATE, DELETE, transactions).
